# Homework 2: Vector Search

This is my homework 2 based on the 
[llm-zoomcamp-2026](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/02-vector-search/homework.md)

## Q1. Embedding a query

In [106]:
from embedder import Embedder

emb = Embedder()

In [107]:
q = "How does approximate nearest neighbor search work?"
v = emb.encode(q)

v[0]

np.float64(-0.02058203437252893)

In [108]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

## Q2. Cosine similarity

In [109]:
vs_docs = [
    doc
    for doc in documents
    if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
][0]

# Vector search embeded vector.
vs_vec = emb.encode(vs_docs["content"])

# Cosine similarity between `vs_vec` and `v`
cos_sim = vs_vec @ v

print(cos_sim)

0.36107027225589694


## Q3. Chunking and searching by hand

In [110]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [111]:
content_batch = [chunk["content"] for chunk in chunks]
x = emb.encode_batch(content_batch)

In [112]:
scores = x @ v
argmax = scores.argmax()
scores[argmax]

chunks[argmax]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

## Q4. Vector search with minsearch

In [113]:
from minsearch import VectorSearch

vs_index = VectorSearch(keyword_fields=["filename"])
vs_index.fit(x, chunks)

In [114]:
query_vec = emb.encode("What metric do we use to evaluate a search engine?")
vs_index.search(query_vec)[0]['filename']

'04-evaluation/lessons/05-search-metrics.md'

## Q5. Text search vs vector search

In [115]:
from minsearch import Index

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(chunks)

In [116]:
query = "How do I store vectors in PostgreSQL?"
query_vec = emb.encode(query)


top_text_results = index.search(query, num_results=5)
top_vec_results = vs_index.search(query_vec, num_results=5)

print(f"{"Top Text Results":<50} Top Vector Results")
for text_result, vec_result in zip(top_text_results, top_vec_results):
    print(f"{text_result["filename"]:<50}", vec_result["filename"]) 


Top Text Results                                   Top Vector Results
02-vector-search/lessons/02-embeddings.md          02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md                 02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/01-intro.md               03-orchestration/lessons/05-rag.md
03-orchestration/lessons/05-rag.md                 02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/01-intro.md               02-vector-search/lessons/08-pgvector.md


The value that appears in vector search but not in text search is 
`02-vector-search/lessons/08-pgvector.md`

## Q6. Hybrid search

In [117]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [118]:
query = "How do I give the model access to tools?"
query_vec = emb.encode(query)

text_results = index.search(query)
vector_results = vs_index.search(query_vec)

results = rrf([vector_results, text_results],num_results=1)

In [119]:
display(results[0]['filename'])

'01-agentic-rag/lessons/13-function-calling.md'